Criar uma tabela de ordens ja com:<br>
quantidade de itens, vendendores, lojas, mapeamento de Status e valor total da venda (valor total - desconto)<br><br>
Mapeamento Status:<br>
1 THEN 'Pending'<br>
2 THEN 'Processing'<br>
3 THEN 'Shipped'<br>
4 THEN 'Delivered'<br>
ELSE 'Unknown'<br>

In [0]:
# Definir pastas do projetos em variaveis para facilitar
bronze_path   = '/Volumes/bikestore/default/bikestore/bronze/'
silver_path   = '/Volumes/bikestore/default/bikestore/silver/'
gold_path     = '/Volumes/bikestore/default/bikestore/gold/'
resource_path = '/Volumes/bikestore/default/bikestore/resource/origem/'

In [0]:
#criando um dicionario com o caminho de cada pasta do arquivo parquet/delta
bronze_map = {
    'tmp_brands': f'{bronze_path}brand/',
    'tmp_customers': f'{bronze_path}customers/',
    'tmp_orders': f'{bronze_path}orders/',
    'tmp_order_items': f'{bronze_path}orders_item/',
    'tmp_products': f'{bronze_path}products/',
    'tmp_stores': f'{bronze_path}stores/',
    'tmp_staff': f'{bronze_path}staffs/',
    'tmp_categories': f'{bronze_path}categories/',
    'tmp_stocks': f'{bronze_path}stocks/'
    }

#fazendo um looping para criar uma tabela temporaria pra cada tabela
for key, value in bronze_map.items():
    (spark.read.format('delta')
        .load(value)
        .createOrReplaceTempView(key)
)

In [0]:
%python
df_orders_silver = spark.sql("""
with order_items as(
SELECT 
   OI.order_id
  ,OI.item_id
  ,OI.product_id
  ,OI.quantity
  ,OI.list_price 
  --,(OI.list_price * OI.quantity) value_test
  ,ROUND((OI.list_price * OI.quantity)* (1-OI.discount),2) AS total_sale
  ,OI.discount
FROM tmp_order_items OI

)

-- select final ''
SELECT 
  ORD.order_id,
  ORD.customer_id,
  CASE 
      WHEN ORD.order_status = 1 THEN 'Pending'
      WHEN ORD.order_status = 2 THEN 'Processing'
      WHEN ORD.order_status = 3 THEN 'Shipped'
      WHEN ORD.order_status = 4 THEN 'Delivered'
      ELSE 'Unknown'
    END status,
  ORD.order_status,
  ORD.order_date,
  ORD.required_date,
  ORD.shipped_date,
  --ORD.store_id,
  --st.store_id AS store_id2store,
  ST.store_name,
  ST.state,
  ST.city,
  --ORD.staff_id,
  --STF.staff_id AS IDSTAF2,
  STF.first_name AS first_name_staff,
  STF.active AS active_staff,
  STF.email,
  OIT.product_id,
  OIT.quantity,
  OIT.total_sale,
  OIT.list_price,
  OIT.discount
FROM   tmp_orders ORD
LEFT JOIN tmp_stores ST  ON ORD.store_id = ST.store_id
LEFT JOIN tmp_staff STF ON ORD.staff_id = STF.staff_id
LEFT JOIN       order_items OIT ON ORD.order_id = OIT.order_id
                              """)

# salvar em Delta na silver 
df_orders_silver.write\
    .mode('overwrite')\
    .format('delta')\
    .option('mergeSchema','true')\
    .save(f'{silver_path}orders')

In [0]:
#criando tabela
df = df_orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bikestore.logistics.silver_orders")